<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/03_dicom_to_nifti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DICOM'dan NIfTI'ye Dönüştürme
ADNI (Alzheimer's Disease Neuroimaging Initiative) verileri, IDA internet sitesinden indirilmiş olup DICOM formatındadır. Bu verilerin görüntü işleme algoritmaları ve makine öğrenimi modelleri tarafından kullanılabilmesi için standart bir format olan NIfTI'ye dönüştürülmesi gerekmektedir. Bu not defteri, bu dönüştürme işlemini gerçekleştirmektedir.

Dönüştürme sürecinde, DICOM'dan NIfTI'ye hızlı ve güvenilir bir şekilde dönüşüm sağlayan, aynı zamanda orijinal görüntü bilgilerini ve metadata'yı koruyan `dcm2niix` aracı kullanılmıştır. Dönüştürme işlemi yapılırken, orijinal veri setinin klasörleme yapısı titizlikle incelenmiş ve elde edilen NIfTI verileri aynı hiyerarşik düzen içerisinde kaydedilmiştir. Bu sayede, veri organizasyonu ve erişilebilirliği sağlanmıştır.

# Neden dcm2niix tercih edildi?
Projelerimizde dcm2niix aracını tercih etmemizin temel nedeni, bu aracın tıbbi görüntüleme dünyasında 'altın standart' olarak kabul edilmesidir. Özellikle farklı üreticilere ait DICOM verilerindeki karmaşık meta verileri (yönelim, dilim kalınlığı vb.) hiçbir veri kaybı olmadan NIfTI formatına aktarabilmektedir. C++ tabanlı yapısı sayesinde büyük ADNI veri setlerini çok hızlı işleyebilmekte ve oluşturduğu hassas koordinat sistemi (sform/qform) sayesinde, sonraki aşamalarda kullandığımız HD-BET ve ANTs gibi gelişmiş araçlarla tam uyumlu çalışmaktadır.

In [ ]:
#KÜTÜPHANE KURULUMU
!apt-get install -y dcm2niix -q
!pip install nibabel -q

In [ ]:

import os
import subprocess
import shutil
import glob
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
#Kullanılan klasörlerin yolu burada belirtilmiştir.
CONFIG_CN = {
    "kategori": "CN",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/dataset/CN/ADNI',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/nifti_dataset/nifti_CN'
}

CONFIG_EMCI = {
    "kategori": "EMCI",
    "kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/dataset/EMCI/ADNI',
    "hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/nifti_dataset/nifti_EMCI'
}

In [ ]:
import os
import subprocess
import shutil
import glob
from google.colab import drive


def dicom_to_nifti_pipeline(config, max_hasta=100):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    kategori = config["kategori"]
    temp_dir = f'/content/temp_{kategori}'

    # İstatistikler
    stats = {'yeni_hasta': 0, 'atlanan_hasta': 0, 'yeni_seans': 0, 'atlanan_seans': 0, 'hatali': 0}

    os.makedirs(output_base, exist_ok=True)
    if os.path.exists(temp_dir): shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    print(f"\n{kategori} kategori için dicomdan niftiye dönüşüm işlemi başlıyor.")
    print("-" * 80)

    if not os.path.exists(source_base):
        print(f"Hata: Yol bulunamadı: {source_base}")
        return

    # Hastalar ID'lerine göre listelenir.
    hastalar = sorted([h for h in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, h))])

    for i, subject_id in enumerate(hastalar):
        if (stats['yeni_hasta'] + stats['atlanan_hasta']) >= max_hasta:
            print(f"\nİstenen hasta limitine ({max_hasta}) ulaşıldı. Kalan {len(hastalar) - i} hasta atlanıyor.")
            break

        # Yol: ADNI / Hasta / MPRAGE
        mprage_path = os.path.join(source_base, subject_id, "MPRAGE")
        if not os.path.exists(mprage_path): continue

        print(f"\n {i+1}. Hasta Klasörü: {subject_id}")#Kaçıncı hastada olduğu belirlenerek dönüşüm işleminin ilerleyişi takip edilmiştir.

        # Seanslar listelenir. (Tarih klasörleri).Bazı hastaların farklı tarihlerde çekilmiş MRI görüntüleri bulunabilir .
        seanslar = sorted(os.listdir(mprage_path))
        yeni_islem_var = False
        tum_seanslar_atlandi = True

        for seans in seanslar:
            seans_yolu = os.path.join(mprage_path, seans)
            if not os.path.isdir(seans_yolu): continue

            # Hedef: HastaID / SeansTarihi /
            hedef_seans_yolu = os.path.join(output_base, subject_id, seans)

            # Burada yapılan atlanan kontrolü ile daha önce dönüştürülmüş hasta ve saans dosyalarının atlanması sağlanmıştır.
            if os.path.exists(hedef_seans_yolu) and any(f.endswith('.nii.gz') for f in os.listdir(hedef_seans_yolu)):
                print(f"   [Atlandı] Seans: {seans}")
                stats['atlanan_seans'] += 1
                continue

            # Seans dosyalarının altında bulunan dicom klasörleri bulunur.
            try:
                seri_no_folder = [f for f in os.listdir(seans_yolu) if os.path.isdir(os.path.join(seans_yolu, f))][0]
                final_dicom_path = os.path.join(seans_yolu, seri_no_folder)
            except: continue

            tum_seanslar_atlandi = False
            os.makedirs(hedef_seans_yolu, exist_ok=True)
            print(f"   [İşleniyor] Seans: {seans} ...", end="")

            # Temp temizle
            for f in os.listdir(temp_dir): os.remove(os.path.join(temp_dir, f))

            try:
                # Dönüştür
                subprocess.run(['dcm2niix', '-z', 'y', '-f', seans, '-o', temp_dir, final_dicom_path], capture_output=True)

                # Sadece ana dosyaları hedef SEANS klasörüne taşı
                found = False
                for f in os.listdir(temp_dir):
                    if f == f"{seans}.nii.gz" or f == f"{seans}.json":
                        shutil.move(os.path.join(temp_dir, f), os.path.join(hedef_seans_yolu, f))
                        if f.endswith('.nii.gz'): found = True
                    else:
                        os.remove(os.path.join(temp_dir, f))

                if found:
                    print("  Başarılı")
                    stats['yeni_seans'] += 1; yeni_islem_var = True
                else:
                    print("  Başarısız")
                    stats['hatali'] += 1
            except:
                print("  Hata"); stats['hatali'] += 1

        if yeni_islem_var: stats['yeni_hasta'] += 1
        elif tum_seanslar_atlandi and len(seanslar) > 0: stats['atlanan_hasta'] += 1

    # Burada yeni dönüştürülen ve daha önce dönüştürülen hastaların ve seansların sayısı tutulur.
    print("\n" + "="*80)
    print(f" {kategori} GRUBU GENEL ÖZETİ")
    print("-" * 80)
    print(f"  - Yeni Dönüştürülen Hasta : {stats['yeni_hasta']}")
    print(f"  - Atlanan (Mevcut) Hasta  : {stats['atlanan_hasta']}")
    print("-" * 40)
    print(f"  - Yeni Dönüştürülen Seans : {stats['yeni_seans']}")
    print(f"  - Atlanan (Mevcut) Seans  : {stats['atlanan_seans']}")
    print(f"  - Hatalı İşlem Sayısı     : {stats['hatali']}")
    print("="*80)

In [ ]:

dicom_to_nifti_pipeline(CONFIG_CN, max_hasta=100)


In [ ]:

dicom_to_nifti_pipeline(CONFIG_EMCI, max_hasta=100)